In [81]:
import numpy as np
import torch.nn as nn

In [82]:
data = np.load("data/processed.npz")

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

X_test = data["X_test"]
y_test = data["y_test"]

In [83]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [84]:
batch_size = 128

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

X_batch, y_batch = next(iter(train_loader))

print("X shape:", X_batch.shape)
print("y shape:", y_batch.shape)

X shape: torch.Size([128, 144, 41])
y shape: torch.Size([128])


In [85]:
import sys
sys.path.append("../models/")
from gru import GRUModel as Model

In [86]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model(input_size=X_train_tensor.shape[2]).to(device)
print(model)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
epochs = 20

GRUModel(
  (gru): GRU(41, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [87]:
import joblib

y_scaler = joblib.load("data/target_scaler.pkl")

In [88]:
import math

train_losses = []
val_losses = []

target_std = y_scaler.scale_[0]

for epoch in range(epochs):
    model.train()
    train_loss = 0
    train_mae = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)
        train_mae += torch.abs(predictions.detach() - y_batch).sum().item()

    model.eval()
    val_loss = 0
    val_mae = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            val_loss += loss.item() * X_batch.size(0)
            val_mae += torch.abs(predictions - y_batch).sum().item()

    train_loss /= len(train_loader.dataset)
    val_loss /= len(val_loader.dataset)

    train_rmse = math.sqrt(train_loss) * target_std
    val_rmse = math.sqrt(val_loss) * target_std

    train_mae = train_mae / len(train_loader.dataset) * target_std
    val_mae = val_mae / len(val_loader.dataset) * target_std

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{epochs} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f} | Train RMSE: {train_rmse:.2f} | Val RMSE: {val_rmse:.2f} | Train MAE: {train_mae:.2f} | Val MAE: {val_mae:.2f}")

Epoch 1/20 | Train MSE: 0.6358 | Val MSE: 0.4084 | Train RMSE: 85.40 | Val RMSE: 68.44 | Train MAE: 45.10 | Val MAE: 32.09
Epoch 2/20 | Train MSE: 0.4680 | Val MSE: 0.3141 | Train RMSE: 73.27 | Val RMSE: 60.02 | Train MAE: 36.40 | Val MAE: 27.97
Epoch 3/20 | Train MSE: 0.3947 | Val MSE: 0.2944 | Train RMSE: 67.29 | Val RMSE: 58.11 | Train MAE: 32.41 | Val MAE: 27.64
Epoch 4/20 | Train MSE: 0.3785 | Val MSE: 0.3026 | Train RMSE: 65.90 | Val RMSE: 58.92 | Train MAE: 31.73 | Val MAE: 29.53
Epoch 5/20 | Train MSE: 0.3694 | Val MSE: 0.2908 | Train RMSE: 65.09 | Val RMSE: 57.76 | Train MAE: 31.13 | Val MAE: 25.97
Epoch 6/20 | Train MSE: 0.3612 | Val MSE: 0.2900 | Train RMSE: 64.36 | Val RMSE: 57.68 | Train MAE: 31.28 | Val MAE: 25.62
Epoch 7/20 | Train MSE: 0.3563 | Val MSE: 0.2827 | Train RMSE: 63.93 | Val RMSE: 56.95 | Train MAE: 30.88 | Val MAE: 25.73
Epoch 8/20 | Train MSE: 0.3483 | Val MSE: 0.2918 | Train RMSE: 63.21 | Val RMSE: 57.85 | Train MAE: 30.67 | Val MAE: 28.20
Epoch 9/20 | Tra